# Modelo Preditivo de NPS — Tech Challenge

## Visão Geral

Neste notebook construímos um **modelo de classificação** capaz de prever se um cliente será **Detrator**, **Neutro** ou **Promotor** com base em variáveis operacionais do pedido.

### Etapas

1. Carregamento dos dados
2. Definição da variável alvo (`nps_class`)
3. Seleção e preparação das features
4. Divisão treino/teste
5. Treinamento do modelo (Random Forest)
6. Avaliação dos resultados

### Instalação de dependências

In [9]:
%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importação das bibliotecas

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### 1. Carregamento dos dados

Carregamos o dataset bruto do Tech Challenge a partir do arquivo CSV.

In [11]:
df = pd.read_csv('../data/raw/desafio_nps_fase_1.csv')
df.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


### 2. Definição da variável alvo (`nps_class`)

Criamos a coluna `nps_class` categorizando cada cliente com base na sua nota NPS:

- **Detrator** (nota 0–6): clientes insatisfeitos
- **Neutro** (nota 7–8): clientes indiferentes
- **Promotor** (nota 9–10): clientes entusiastas

Essa será a variável que o modelo tentará prever.

In [12]:
def categorize_nps(score):
    if score <= 6:
        return 'Detrator'
    elif score <= 8:
        return 'Neutro'
    else:
        return 'Promotor'

df['nps_class'] = df['nps_score'].apply(categorize_nps)

df['nps_class'].value_counts()

nps_class
Detrator    1851
Neutro       448
Promotor     201
Name: count, dtype: int64

### 3. Seleção e preparação das features

Removemos colunas que não agregam valor preditivo ao modelo:
- `customer_id` e `order_id` — são identificadores únicos, sem poder preditivo
- `nps_score` — é a origem da variável alvo; usá-la seria vazamento de dados (*data leakage*)
- `nps_class` — é a própria variável alvo

A coluna `customer_region` é textual, por isso aplicamos **One-Hot Encoding** via `pd.get_dummies`.

Valores ausentes são preenchidos com a **mediana** de cada coluna para evitar erros no treinamento.

In [13]:
X = df.drop(columns=['customer_id', 'order_id', 'nps_score', 'nps_class'])
y = df['nps_class']

X = pd.get_dummies(X, columns=['customer_region'], drop_first=True)
X = X.fillna(X.median())

print(f'Features: {X.shape[1]} colunas | Amostras: {X.shape[0]} linhas')
X.head()

Features: 19 colunas | Amostras: 2500 linhas


,customer_age,customer_tenure_months,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,repeat_purchase_30d,complaints_count,csat_internal_score,customer_region_Nordeste,customer_region_Norte,customer_region_Sudeste,customer_region_Sul
0,63,14,139.73,4,39.35,4,2,2,55.53,3,0,4,0,3,6.5,True,False,False,False
1,20,1,458.95,2,9.51,10,6,4,28.23,3,0,10,0,3,0.0,False,False,False,True
2,46,111,507.06,5,42.82,6,6,1,40.99,1,4,5,0,7,1.5,True,False,False,False
3,52,117,302.19,2,19.58,9,5,2,35.24,3,1,11,0,4,0.3,False,False,False,False
4,56,50,253.06,1,29.37,11,13,1,39.32,1,1,0,0,3,7.9,False,True,False,False


### 4. Divisão treino / teste

Dividimos os dados em:
- **80% treino** — usado para o modelo aprender os padrões
- **20% teste** — usado para validar o desempenho em dados nunca vistos

O parâmetro `stratify=y` garante que a proporção de cada classe (Detrator, Neutro, Promotor) seja mantida igualmente nos dois conjuntos.

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

Treino: 2000 amostras
Teste:  500 amostras


### 5. Treinamento do modelo — Random Forest

Optamos pelo **Random Forest Classifier** pelos seguintes motivos:

- Lida bem com múltiplas variáveis operacionais ao mesmo tempo
- Menor risco de *overfitting* em comparação a uma árvore de decisão simples
- Não exige normalização dos dados
- Fornece a importância de cada feature nativamente

In [15]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print('Modelo treinado com sucesso!')

Modelo treinado com sucesso!


### 6. Avaliação dos resultados

Avaliamos o modelo no conjunto de teste usando o **relatório de classificação**, que apresenta para cada classe:

- **Precision** — dos que o modelo previu como X, quantos realmente eram X
- **Recall** — dos que realmente eram X, quantos o modelo acertou
- **F1-score** — média harmônica entre precision e recall
- **Support** — número de amostras reais de cada classe no conjunto de teste

In [16]:
y_pred = model.predict(X_test)

print('Relatório de Classificação (Performance do Modelo):')
print(classification_report(y_test, y_pred))

Relatório de Classificação (Performance do Modelo):
              precision    recall  f1-score   support

    Detrator       0.87      0.96      0.91       370
      Neutro       0.67      0.37      0.47        90
    Promotor       0.93      1.00      0.96        40

    accuracy                           0.85       500
   macro avg       0.82      0.77      0.78       500
weighted avg       0.84      0.85      0.84       500



### 7 Gerador para leitura de dockertfile

In [ ]:
import joblib

model.fit(X_train, y_train)

# Salva o modelo treinado na pasta models
joblib.dump(model, '../models/modelo_nps.pkl')
print("Modelo salvo com sucesso!")